In [28]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [29]:
# 데이터 로드
df = pd.read_csv('extracted_data.csv')

# ----------------------------------
# 1. 마지막 조사 데이터 제거
# ----------------------------------

df = df[df['churn'].notna()]

In [30]:
# ----------------------------------
# 파생변수 생성
# ----------------------------------

df["installment_ratio"] = (
    df["monthly_installment"] /
    (df["monthly_total_cost"] + 1)
) # 할부금 비율

df["cost_income_ratio"] = (
    df["monthly_total_cost"] /
    (df["income"] + 1)
) # 통신비 대비 소득

df["income_per_person"] = (
    df["income"] /
    df["household_size"]
) # 1인당 소득

df["cost_per_person"] = (
    df["monthly_total_cost"] /
    df["household_size"]
) # 1인당 통신비

df["married_large_family"] = (
    (df["marriage"] == 2) &
    (df["household_size"] >= 3)
).astype(int) # 결혼 + 가구원수

In [31]:
# ----------------------------------
# 2. 변수 구분
# ----------------------------------

numeric_features = [
    'year',
    'age',
    'income',
    'monthly_total_cost',
    'monthly_installment',
    'household_size',
    'installment_ratio',
    'cost_income_ratio',
    'income_per_person',
    'cost_per_person'
]

categorical_features = [
    'gender',
    'school',
    'area',
    'job',
    'marriage',
    'cost_payer',
    'provider',
    'married_large_family',
    'is_mobile_bundled'
]

# ----------------------------------
# 3. 수치형 파이프라인
# ----------------------------------

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# ----------------------------------
# 4. 범주형 파이프라인
# ----------------------------------

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# ----------------------------------
# 5. 전체 전처리
# ----------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [32]:
# --------------------------
# 6. 연도 기준 분할
# --------------------------

train_df = df[df['year'] <= 20]

valid_df = df[
    (df['year'] >= 21) &
    (df['year'] <= 22)
]

test_df = df[df['year'] >= 23]

print(len(train_df))
print(len(valid_df))
print(len(test_df))

91378
18823
8135


In [33]:
# --------------------------
# 7. 전처리 적용
# --------------------------

X_train = train_df.drop(columns=['id', 'churn'])
y_train = train_df['churn']

X_valid = valid_df.drop(columns=['id', 'churn'])
y_valid = valid_df['churn']

X_test = test_df.drop(columns=['id', 'churn'])
y_test = test_df['churn']

# 전처리 적용
X_train = preprocessor.fit_transform(X_train)

X_valid = preprocessor.transform(X_valid)

X_test = preprocessor.transform(X_test)

In [34]:
# --------------------------
# 평가 함수 정의
# --------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X)[:, 1]
    else:
        y_prob = model.predict(X)

    print(f"\n[{dataset_name}]")
    print("Accuracy :", round(accuracy_score(y, y_pred), 4))
    print("Precision:", round(precision_score(y, y_pred), 4))
    print("Recall   :", round(recall_score(y, y_pred), 4))
    print("F1-score :", round(f1_score(y, y_pred), 4))
    print("ROC-AUC  :", round(roc_auc_score(y, y_prob), 4))

In [35]:
# --------------------------
# LightGBM 학습
# --------------------------

from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    objective='binary',
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=-1,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=42,
    class_weight='balanced'
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)]
)

print('train finish')

[LightGBM] [Info] Number of positive: 33970, number of negative: 57408
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009085 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1119
[LightGBM] [Info] Number of data points in the train set: 91378, number of used features: 54
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
train finish


In [36]:
# --------------------------
# LightGBM 평가
# --------------------------

evaluate_model(lgb_model, X_train, y_train, "Train")
evaluate_model(lgb_model, X_valid, y_valid, "Validation")
evaluate_model(lgb_model, X_test, y_test, "Test")

C:\unit_project_workspace\unit_project_workspace\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\unit_project_workspace\unit_project_workspace\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[Train]
Accuracy : 0.6916
Precision: 0.5724
Recall   : 0.673
F1-score : 0.6187
ROC-AUC  : 0.7523


C:\unit_project_workspace\unit_project_workspace\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\unit_project_workspace\unit_project_workspace\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[Validation]
Accuracy : 0.6041
Precision: 0.4635
Recall   : 0.5957
F1-score : 0.5213
ROC-AUC  : 0.6481


C:\unit_project_workspace\unit_project_workspace\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\unit_project_workspace\unit_project_workspace\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[Test]
Accuracy : 0.5889
Precision: 0.4705
Recall   : 0.5499
F1-score : 0.5071
ROC-AUC  : 0.6278


In [37]:
# --------------------------
# LightGBM 중요 변수 확인
# --------------------------

import pandas as pd
import matplotlib.pyplot as plt

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": lgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print(importance_df.head(20))

                       feature  importance
7       num__cost_income_ratio        8268
9         num__cost_per_person        7124
3      num__monthly_total_cost        6761
0                    num__year        6621
6       num__installment_ratio        6530
4     num__monthly_installment        4058
1                     num__age        3731
8       num__income_per_person        2242
2                  num__income        1585
47           cat__provider_1.0         736
54  cat__is_mobile_bundled_1.0         662
18               cat__area_1.0         608
25               cat__area_8.0         597
16             cat__school_5.0         591
48           cat__provider_2.0         588
49           cat__provider_3.0         578
15             cat__school_4.0         568
38           cat__marriage_2.0         500
5          num__household_size         487
13             cat__school_2.0         480


In [38]:
# --------------------------
# XGBoost 학습
# --------------------------

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    gamma=1,
    reg_alpha=1,
    reg_lambda=2,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

print('train finish')

train finish


In [39]:
# --------------------------
# XGBoost 평가
# --------------------------

evaluate_model(xgb_model, X_train, y_train, "Train")
evaluate_model(xgb_model, X_valid, y_valid, "Validation")
evaluate_model(xgb_model, X_test, y_test, "Test")


[Train]
Accuracy : 0.672
Precision: 0.6441
Recall   : 0.263
F1-score : 0.3735
ROC-AUC  : 0.692

[Validation]
Accuracy : 0.6596
Precision: 0.5623
Recall   : 0.2682
F1-score : 0.3632
ROC-AUC  : 0.6501

[Test]
Accuracy : 0.6377
Precision: 0.5678
Recall   : 0.2423
F1-score : 0.3397
ROC-AUC  : 0.6339


In [40]:
# --------------------------
# XGBoost 중요 변수 확인
# --------------------------

feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

print(importance_df.head(20))

                       feature  importance
47           cat__provider_1.0    0.155034
30              cat__area_13.0    0.065009
29              cat__area_12.0    0.064599
22               cat__area_5.0    0.062856
18               cat__area_1.0    0.044535
20               cat__area_3.0    0.035425
51           cat__provider_5.0    0.032761
0                    num__year    0.025931
31              cat__area_14.0    0.021764
26               cat__area_9.0    0.019713
32              cat__area_15.0    0.019030
49           cat__provider_3.0    0.018893
50           cat__provider_4.0    0.018539
33              cat__area_16.0    0.017994
21               cat__area_4.0    0.016555
24               cat__area_7.0    0.016178
27              cat__area_10.0    0.015347
1                     num__age    0.015324
54  cat__is_mobile_bundled_1.0    0.013842
48           cat__provider_2.0    0.013740


In [41]:

# --------------------------
# 서비스용 파이프라인 모델 학습 및 저장
# --------------------------

import joblib
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# preprocessor는 위에서 이미 fit_transform 완료된 상태이므로 새로 만들어 fit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features_s = [
    'year', 'age', 'income', 'monthly_total_cost',
    'monthly_installment', 'household_size',
    'installment_ratio', 'cost_income_ratio',
    'income_per_person', 'cost_per_person'
]

categorical_features_s = [
    'gender', 'school', 'area', 'job', 'marriage',
    'cost_payer', 'provider', 'married_large_family', 'is_mobile_bundled'
]

preprocessor_s = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features_s),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features_s),
])

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor_s),
    ('model', XGBClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        gamma=1,
        reg_alpha=1,
        reg_lambda=2,
        eval_metric='auc',
        random_state=42,
    )),
])

FEATURE_COLS_S = numeric_features_s + categorical_features_s

X_tr = train_df[FEATURE_COLS_S]
y_tr = train_df['churn']
X_va = valid_df[FEATURE_COLS_S]
y_va = valid_df['churn']
X_te = test_df[FEATURE_COLS_S]
y_te = test_df['churn']

xgb_pipeline.fit(X_tr, y_tr)
print("학습 완료")

evaluate_model(xgb_pipeline, X_tr, y_tr, "Train")
evaluate_model(xgb_pipeline, X_va, y_va, "Validation")
evaluate_model(xgb_pipeline, X_te, y_te, "Test")

Path('../models').mkdir(exist_ok=True)
joblib.dump(xgb_pipeline, '../models/xgb_pipeline.joblib')
print("\n모델 저장 완료: models/xgb_pipeline.joblib")


학습 완료

[Train]
Accuracy : 0.672
Precision: 0.6441
Recall   : 0.263
F1-score : 0.3735
ROC-AUC  : 0.692

[Validation]
Accuracy : 0.6596
Precision: 0.5623
Recall   : 0.2682
F1-score : 0.3632
ROC-AUC  : 0.6501

[Test]
Accuracy : 0.6377
Precision: 0.5678
Recall   : 0.2423
F1-score : 0.3397
ROC-AUC  : 0.6339

모델 저장 완료: models/xgb_pipeline.joblib
